# Chains in LangChain

> **Revision Goal:** Understand what a Chain is, why it is needed, how data flows through a chain, LCEL, sequential/parallel/conditional chains, and how chains are used in real GenAI/RAG applications.

---

# 1. What is a Chain?

A **Chain** is a workflow in which multiple components are connected together to perform a task.

The output of one component becomes the input of the next component.

### Basic Idea

    Input
      ↓
    Component 1
      ↓
    Component 2
      ↓
    Component 3
      ↓
    Output

### Example

    User Input
        ↓
    Prompt Template
        ↓
    LLM
        ↓
    Output Parser
        ↓
    Final Answer

### Simple Definition

> **Chain = Multiple connected operations executed as one workflow.**

---

# 2. Why Do We Need Chains?

An LLM application normally requires multiple steps.

For example, suppose we want to ask an LLM:

> Explain Artificial Intelligence in simple words.

We may need:

    User Input
        ↓
    Create Prompt
        ↓
    Send Prompt to LLM
        ↓
    Receive LLM Response
        ↓
    Parse Response
        ↓
    Return Final Answer

Without a chain, we have to manually execute every step.

With a chain:

    Prompt → LLM → Parser

We can execute the entire workflow with one call.

---

# 3. Chain Without LangChain Composition

Without composing the components:

    input
      ↓
    prompt.invoke()
      ↓
    llm.invoke()
      ↓
    parser.invoke()
      ↓
    output

Conceptually:

    prompt_output = prompt.invoke(input)

    llm_output = llm.invoke(prompt_output)

    final_output = parser.invoke(llm_output)

Here, we manually pass the output of each step to the next step.

---

# 4. Chain Using LCEL

LangChain provides **LCEL (LangChain Expression Language)** for composing components.

The most important operator is:

    |

This is called the **pipe operator**.

Example:

    chain = prompt | llm | parser

It means:

    Input
      ↓
    Prompt
      ↓
    LLM
      ↓
    Parser
      ↓
    Output

Instead of manually connecting every component, LCEL allows us to express the complete workflow clearly.

---

# 5. Most Important Concept: Output → Input

This is the most important concept to understand about chains.

    Component A
         ↓
      Output A
         ↓
    Component B
         ↓
      Output B
         ↓
    Component C
         ↓
      Output C

In other words:

    Output of A = Input of B

    Output of B = Input of C

Example:

    Prompt
      ↓
    PromptValue
      ↓
    LLM
      ↓
    AIMessage
      ↓
    Parser
      ↓
    String

Therefore:

    prompt | llm | parser

represents the flow of data between the components.

---

# 6. Basic Chain Example

    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser

    prompt = ChatPromptTemplate.from_template(
        "Explain {topic} in simple words."
    )

    llm = ChatOpenAI()

    parser = StrOutputParser()

    chain = prompt | llm | parser

    result = chain.invoke({
        "topic": "Artificial Intelligence"
    })

    print(result)

### Flow

    {"topic": "Artificial Intelligence"}
                    ↓
             Prompt Template
                    ↓
                   LLM
                    ↓
            StrOutputParser
                    ↓
              Final String

---

# 7. What Happens Internally?

When we write:

    chain = prompt | llm | parser

and execute:

    chain.invoke({
        "topic": "AI"
    })

the conceptual flow is:

    Input
      ↓
    prompt.invoke(input)
      ↓
    Prompt Output
      ↓
    llm.invoke(prompt_output)
      ↓
    LLM Output
      ↓
    parser.invoke(llm_output)
      ↓
    Final Output

The chain handles the connection between these operations.

---

# 8. Chain as a Pipeline

A very useful mental model is to think of a Chain as a **pipeline**.

    ┌────────────┐
    │   Input    │
    └─────┬──────┘
          ↓
    ┌────────────┐
    │   Prompt   │
    └─────┬──────┘
          ↓
    ┌────────────┐
    │    LLM     │
    └─────┬──────┘
          ↓
    ┌────────────┐
    │   Parser   │
    └─────┬──────┘
          ↓
    ┌────────────┐
    │   Output   │
    └────────────┘

Each stage performs a specific operation.

---

# 9. Components Commonly Used in Chains

A chain can contain many different components:

    Prompt Template
          ↓
    Chat Model / LLM
          ↓
    Output Parser
          ↓
    Retriever
          ↓
    RunnableLambda
          ↓
    RunnablePassthrough
          ↓
    RunnableParallel
          ↓
    RunnableBranch
          ↓
    Another Chain

The important thing is that the components must be compatible with each other's inputs and outputs.

---

# 10. Runnable

Modern LangChain composition is based heavily on **Runnables**.

A Runnable is a component that can be executed and composed with other Runnables.

Examples:

    Prompt
    LLM
    Chat Model
    Output Parser
    Retriever
    RunnableLambda
    RunnablePassthrough
    RunnableParallel
    RunnableBranch
    Another Chain

Think:

    Runnable → Runnable → Runnable

---

# 11. RunnableSequence

A **RunnableSequence** represents sequential execution.

Flow:

    A
    ↓
    B
    ↓
    C
    ↓
    D

Each component runs after the previous component.

Example:

    prompt | llm | parser

Conceptually:

    Prompt
      ↓
    LLM
      ↓
    Parser

### Key Point

> **Sequence = One step after another.**

---

# 12. Sequential Chain

A sequential chain is useful when the next step depends on the previous step.

Example:

    User Input
        ↓
    Generate Topic
        ↓
    Generate Explanation
        ↓
    Generate Summary
        ↓
    Final Output

Here:

    Step 2 depends on Step 1

and:

    Step 3 depends on Step 2

Therefore, the steps must execute sequentially.

---

# 13. Sequential Chain Example

Suppose:

    chain1 = Topic Generation
    chain2 = Explanation Generation
    chain3 = Summary Generation

Then:

    final_chain = chain1 | chain2 | chain3

Flow:

    Input
      ↓
    Chain 1
      ↓
    Chain 2
      ↓
    Chain 3
      ↓
    Output

This is useful for multi-step workflows.

---

# 14. RunnableParallel

Sometimes two or more operations do not depend on each other.

In that case, we can execute them in parallel.

### Sequential

    Input
      ↓
      A
      ↓
      B

### Parallel

            ┌──→ A
    Input ──┤
            └──→ B

The two branches can independently process the same input.

---

# 15. RunnableParallel Example

Example:

    RunnableParallel(
        question=RunnablePassthrough(),
        context=retriever
    )

Flow:

                      ┌──→ Question
                      │
    User Input ───────┤
                      │
                      └──→ Retriever → Context

Output conceptually:

    {
        "question": "What is AI?",
        "context": "Retrieved documents..."
    }

### Key Point

> **Parallel = Multiple independent operations from the same input.**

---

# 16. RunnablePassthrough

`RunnablePassthrough` passes the input forward without modifying it.

Flow:

    Input
      ↓
    RunnablePassthrough
      ↓
    Same Input

Example:

    from langchain_core.runnables import RunnablePassthrough

    chain = RunnablePassthrough()

    result = chain.invoke("Hello")

    Output:

    Hello

### Why is it useful?

It is useful when we need to preserve the original input while also performing another operation.

---

# 17. RunnablePassthrough in RAG

Suppose the user asks:

    What is Machine Learning?

We need two things:

    1. Original question
    2. Retrieved context

We can create:

    {
        "question": RunnablePassthrough(),
        "context": retriever
    }

Flow:

                         ┌──→ Original Question
                         │
    User Question ───────┤
                         │
                         └──→ Retriever
                                  ↓
                               Context

Final structure:

    {
        "question": "What is Machine Learning?",
        "context": "Relevant documents..."
    }

This is one of the most important patterns in RAG chains.

---

# 18. RunnableLambda

`RunnableLambda` allows us to use our own Python function inside a LangChain workflow.

Flow:

    Input
      ↓
    Python Function
      ↓
    Output

Example:

    from langchain_core.runnables import RunnableLambda

    def word_count(text):
        return len(text.split())

    chain = RunnableLambda(word_count)

    result = chain.invoke(
        "LangChain is powerful"
    )

Result:

    3

---

# 19. Why RunnableLambda?

It allows custom logic to become part of a chain.

For example:

    Input
      ↓
    Clean Data
      ↓
    Custom Python Function
      ↓
    Prompt
      ↓
    LLM
      ↓
    Parser
      ↓
    Output

Possible uses:

- Data cleaning
- Data transformation
- Validation
- Custom calculations
- Formatting
- Preprocessing
- Postprocessing
- Business logic

---

# 20. RunnableBranch

`RunnableBranch` is used when the workflow depends on a condition.

It is conceptually similar to:

    if
    elif
    else

in Python.

Flow:

                       ┌──→ Chain A
                       │
    Input → Condition ──┼──→ Chain B
                       │
                       └──→ Default Chain

Example:

    Question
       ↓
    Determine Type
       ↓
    ┌───────────────┬───────────────┐
    ↓               ↓               ↓
    Python          SQL             General
    Chain           Chain           Chain

### Key Point

> **Branch = Choose which path to execute.**

---

# 21. Sequential vs Parallel vs Branch

| Type | Meaning | Diagram |
|---|---|---|
| Sequence | Steps execute one after another | A → B → C |
| Parallel | Independent operations execute together | A → B and C |
| Branch | Select one path based on condition | A → B/C/D |

### Easy Memory Trick

    Sequence = Follow the road

    Parallel = Split into multiple roads

    Branch = Choose one road

---

# 22. Output Parsers in Chains

An LLM may return a model-specific message object.

For example:

    AIMessage(...)

But the application may need:

    String

An output parser converts the model output.

Flow:

    LLM
      ↓
    AIMessage
      ↓
    Output Parser
      ↓
    String

Example:

    parser = StrOutputParser()

    chain = prompt | llm | parser

---

# 23. Why Output Parsers Are Important

Output parsers make LLM output easier for applications to consume.

Possible output formats include:

    String
    JSON
    Dictionary
    Pydantic Object
    Structured Data

Example:

    LLM
      ↓
    Structured Output Parser
      ↓
    Application-friendly Object

This becomes especially important in production applications.

---

# 24. Chain Input and Output

Every chain has:

    Input → Processing → Output

Example:

    Input:

    {
        "topic": "AI"
    }

    ↓

    Prompt

    ↓

    LLM

    ↓

    Parser

    ↓

    Output:

    "Artificial Intelligence is..."

The input expected by the first component and the output produced by the final component define the overall chain interface.

---

# 25. Type Compatibility

One of the most important debugging concepts is **type compatibility**.

Suppose:

    Prompt → LLM → Parser

Conceptually:

    Dictionary
        ↓
    PromptValue / Messages
        ↓
    AIMessage
        ↓
    String

The output of one component must be compatible with what the next component expects.

### Remember

    Output of A
         ↓
    must be compatible with
         ↓
    Input of B

If not, the chain can fail.

---

# 26. Common Chain Error: Wrong Input Key

Suppose:

    prompt = ChatPromptTemplate.from_template(
        "Explain {topic}"
    )

The required key is:

    topic

Correct:

    chain.invoke({
        "topic": "AI"
    })

Incorrect:

    chain.invoke({
        "subject": "AI"
    })

### Rule

If prompt contains:

    {topic}

input should contain:

    "topic"

---

# 27. Common Chain Error: Incompatible Components

Suppose:

    Component A → Dictionary

but:

    Component B → expects String

Then:

    A → B

may fail.

Always check:

    What does this component output?

and:

    What does the next component expect?

This is one of the best ways to debug LCEL chains.

---

# 28. Common Chain Error: Parser Failure

Suppose a parser expects:

    {
        "name": "...",
        "age": 25
    }

But the LLM produces:

    My name is John and I am 25 years old.

The parser may fail because the output does not follow the expected structure.

Therefore, structured output requires:

    Prompt
       ↓
    LLM
       ↓
    Correct Format
       ↓
    Parser
       ↓
    Structured Object

---

# 29. Chain Execution Methods

LangChain Runnables commonly support several execution patterns.

## `invoke()`

Used for one input.

    result = chain.invoke(input)

Flow:

    Input → Chain → Output

---

## `batch()`

Used for multiple inputs.

    results = chain.batch([
        input1,
        input2,
        input3
    ])

Flow:

    Input 1 ─┐
    Input 2 ─┼→ Chain → Outputs
    Input 3 ─┘

---

## `stream()`

Used when output should be received incrementally.

    for chunk in chain.stream(input):
        print(chunk)

Flow:

    Input
      ↓
    Chain
      ↓
    Chunk 1
    Chunk 2
    Chunk 3
    Chunk 4

---

## `ainvoke()`

Asynchronous version of `invoke()`.

    result = await chain.ainvoke(input)

---

## `abatch()`

Asynchronous batch execution.

---

## `astream()`

Asynchronous streaming.

---

# 30. Execution Methods — Revision Table

| Method | Purpose |
|---|---|
| `invoke()` | Execute one input |
| `batch()` | Execute multiple inputs |
| `stream()` | Receive output incrementally |
| `ainvoke()` | Async single execution |
| `abatch()` | Async batch execution |
| `astream()` | Async streaming |

### Easy Memory

    invoke  → one
    batch   → many
    stream  → chunks
    a       → asynchronous

---

# 31. RAG Chain

Chains are extremely important in **RAG (Retrieval-Augmented Generation)**.

A basic RAG workflow is:

    User Question
          ↓
      Retriever
          ↓
    Relevant Context
          ↓
    Prompt Template
          ↓
         LLM
          ↓
       Parser
          ↓
    Final Answer

---

# 32. RAG Chain Detailed Diagram

                         USER QUESTION
                               │
                 ┌─────────────┴─────────────┐
                 │                           │
                 ↓                           ↓
          Original Question             Retriever
                 │                           │
                 │                           ↓
                 │                    Relevant Documents
                 │                           │
                 └─────────────┬─────────────┘
                               ↓
                        Prompt Template
                               ↓
                              LLM
                               ↓
                         Output Parser
                               ↓
                         Final Answer

This is a very important real-world chain architecture.

---

# 33. RAG Chain Using LCEL

Conceptually:

    chain = (
        {
            "context": retriever,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | parser
    )

Execution:

    User Question
         ↓
    ┌───────────────────────────┐
    │                           │
    ↓                           ↓
    Question                Retriever
    │                           │
    │                           ↓
    │                        Context
    │                           │
    └─────────────┬─────────────┘
                  ↓
               Prompt
                  ↓
                 LLM
                  ↓
               Parser
                  ↓
              Answer

---

# 34. Chain Composition

A major advantage of LangChain is that chains can be composed.

Small chains:

    Chain A
    Chain B
    Chain C

can become:

    Chain A → Chain B → Chain C

Example:

    final_chain = chain1 | chain2 | chain3

This allows large applications to be constructed from smaller components.

---

# 35. Nested Chains

A large application can contain multiple smaller chains.

Example:

    Main Application
          │
          ├── Query Processing Chain
          │
          ├── Retrieval Chain
          │
          ├── Generation Chain
          │
          └── Output Processing Chain

This improves:

- Modularity
- Reusability
- Testing
- Debugging
- Maintenance

---

# 36. Chain vs Agent

This distinction is important.

## Chain

A chain generally follows a predefined workflow.

    A
    ↓
    B
    ↓
    C
    ↓
    D

The developer defines the flow.

## Agent

An agent can dynamically decide what action to take.

    User
      ↓
    Agent
      ↓
    Decide
      ↓
    Tool
      ↓
    Observe
      ↓
    Decide Again
      ↓
    Final Answer

### Easy Difference

    Chain
    = Fixed / predefined workflow

    Agent
    = Dynamic / decision-making workflow

---

# 37. Chain vs Normal Python Function

A normal Python workflow may look like:

    def process(data):
        result1 = step1(data)
        result2 = step2(result1)
        return result2

LangChain provides a composable approach:

    chain = step1 | step2

The LangChain Runnable abstraction additionally supports execution patterns such as:

    invoke()
    batch()
    stream()
    ainvoke()
    abatch()
    astream()

and allows integration with other LangChain components.

---

# 38. Benefits of Chains

### 1. Modularity

Break a large task into smaller components.

### 2. Reusability

Reuse the same chain in different places.

### 3. Readability

A workflow such as:

    prompt | llm | parser

is easy to understand.

### 4. Composition

Small workflows can be combined into larger workflows.

### 5. Streaming

Output can be streamed.

### 6. Batch Processing

Multiple inputs can be processed.

### 7. Async Execution

Applications can use asynchronous execution.

### 8. Integration

Chains can connect:

    Prompts
    LLMs
    Chat Models
    Retrievers
    Parsers
    Python Functions
    Other Runnables
    Other Chains

---

# 39. Most Important Chain Patterns

## Pattern 1: Basic LLM Chain

    Input
      ↓
    Prompt
      ↓
    LLM
      ↓
    Output

---

## Pattern 2: LLM + Parser

    Input
      ↓
    Prompt
      ↓
    LLM
      ↓
    Parser
      ↓
    Output

---

## Pattern 3: Sequential Chain

    Input
      ↓
    Chain A
      ↓
    Chain B
      ↓
    Chain C
      ↓
    Output

---

## Pattern 4: Parallel Chain

              ┌→ Chain A
    Input ────┤
              └→ Chain B

---

## Pattern 5: Conditional Chain

    Input
      ↓
    Condition
      ├──→ Chain A
      ├──→ Chain B
      └──→ Default Chain

---

## Pattern 6: RAG Chain

    Question
       ↓
    Retriever
       ↓
    Context
       ↓
    Prompt
       ↓
    LLM
       ↓
    Parser
       ↓
    Answer

---

# 40. Real-World Chain Architecture

A production GenAI application may look like:

    User
      ↓
    Input Processing
      ↓
    Query Transformation
      ↓
    Retriever
      ↓
    Context Processing
      ↓
    Prompt Construction
      ↓
    LLM
      ↓
    Output Parser
      ↓
    Validation
      ↓
    Final Response

Not every application needs all these steps, but chains allow these operations to be connected into a workflow.

---

# 41. Important Mental Model

Whenever you see:

    A | B | C

think:

    Input
      ↓
      A
      ↓
      B
      ↓
      C
      ↓
    Output

The pipe operator means:

> **Pass the result of the left side into the right side.**

---

# 42. Chain Architecture in One Diagram

    ┌──────────────┐
    │     Input    │
    └──────┬───────┘
           ↓
    ┌──────────────┐
    │    Prompt    │
    └──────┬───────┘
           ↓
    ┌──────────────┐
    │     LLM      │
    └──────┬───────┘
           ↓
    ┌──────────────┐
    │ Output Parser│
    └──────┬───────┘
           ↓
    ┌──────────────┐
    │    Output    │
    └──────────────┘

This is the basic mental model of a LangChain chain.

---

# 43. Complete Concept Map

                         CHAINS
                            │
          ┌─────────────────┼─────────────────┐
          │                 │                 │
          ↓                 ↓                 ↓
       LCEL             Runnables          Workflow
          │                 │                 │
          ↓          ┌──────┼──────┐          ↓
       `|`           ↓      ↓      ↓      Input → Output
                    Seq   Parallel Branch
                     │      │       │
                     ↓      ↓       ↓
                    A→B   A→B/C   A→B/C
                            │
                            ↓
                    Components
                            │
        ┌───────────┬───────┼────────┬───────────┐
        ↓           ↓       ↓        ↓           ↓
      Prompt       LLM    Parser  Retriever   Functions
        │           │       │        │           │
        └───────────┴───────┴────────┴───────────┘
                            │
                            ↓
                         RAG
                            │
                            ↓
              Question → Retrieve → Prompt
                            ↓
                           LLM
                            ↓
                         Parser
                            ↓
                         Answer

---

# 44. Quick Revision Table

| Concept | Meaning |
|---|---|
| Chain | Connected workflow of components |
| LCEL | LangChain Expression Language |
| `|` | Pipe operator used for composition |
| Runnable | Executable/composable LangChain component |
| RunnableSequence | Sequential execution |
| RunnableParallel | Parallel execution |
| RunnablePassthrough | Pass input unchanged |
| RunnableLambda | Run custom Python function |
| RunnableBranch | Conditional routing |
| `invoke()` | Execute one input |
| `batch()` | Execute multiple inputs |
| `stream()` | Receive output incrementally |
| `ainvoke()` | Async execution |
| Output Parser | Converts model output |
| RAG Chain | Retrieval + Prompt + LLM + Output processing |

---

# 45. Key Differences for Revision

| Concept | Main Idea |
|---|---|
| Sequence | A → B → C |
| Parallel | A → B and C |
| Branch | Choose A or B or C |
| Passthrough | Keep original input |
| Lambda | Execute custom Python logic |
| Parser | Convert model output |
| Retriever | Retrieve relevant information |
| Chain | Combine multiple operations |

---

# 46. The Most Important Syntax to Remember

    chain = prompt | llm | parser

Think:

    prompt
      ↓
    llm
      ↓
    parser

Then:

    result = chain.invoke(input)

This is the fundamental LCEL chain pattern.

---

# 47. Final Concept Summary

### Chain

    Multiple connected operations

### LCEL

    Language used to compose LangChain components

### Pipe Operator

    |

    Means:

    Output of Left
          ↓
    Input of Right

### Sequential

    A → B → C

### Parallel

        ┌→ B
    A ──┤
        └→ C

### Branch

    Input
      ↓
    Condition
      ├→ A
      ├→ B
      └→ C

### Passthrough

    Input → Same Input

### Lambda

    Input → Custom Python Function → Output

### RAG

    Question
       ↓
    Retriever
       ↓
    Context
       ↓
    Prompt
       ↓
    LLM
       ↓
    Parser
       ↓
    Answer

---

# 48. One-Line Definition for Interview/Revision

> **A Chain in LangChain is a composable workflow that connects multiple Runnables, where the output of one component is passed to the next component to transform an input into a final output.**

---

# 49. Final Memory Trick

Remember:

    CHAIN
      ↓
    CONNECT
      ↓
    COMPONENTS
      ↓
    INPUT → PROCESS → OUTPUT

And for LCEL:

    A | B | C

means:

    A → B → C

For a basic LLM application:

    Input
      ↓
    Prompt
      ↓
    LLM
      ↓
    Parser
      ↓
    Output

For RAG:

    Question
       ↓
    Retrieve
       ↓
    Context
       ↓
    Prompt
       ↓
    LLM
       ↓
    Parser
       ↓
    Answer

> **If you understand these diagrams and the data flow between them, you understand the core concept of Chains in LangChain.**